# Advanced Titanic Competition Pipeline

This notebook implements a competition-grade machine learning pipeline for improving Kaggle leaderboard performance.

Techniques Used:

- Advanced Feature Engineering
- One-Hot Encoding
- Stratified K-Fold Cross Validation
- CatBoost
- XGBoost
- LightGBM
- Ensemble Blending

Goal:
Improve leaderboard score toward 0.85+

In [2]:
! pip install catboost

  Using cached catboost-1.2.10-cp313-cp313-win_amd64.whl.metadata (1.5 kB)
  Using cached graphviz-0.21-py3-none-any.whl.metadata (12 kB)
  Using cached plotly-6.7.0-py3-none-any.whl.metadata (8.6 kB)
  Using cached narwhals-2.21.0-py3-none-any.whl.metadata (16 kB)
Using cached catboost-1.2.10-cp313-cp313-win_amd64.whl (100.2 MB)
Using cached graphviz-0.21-py3-none-any.whl (47 kB)
Using cached plotly-6.7.0-py3-none-any.whl (9.9 MB)
Using cached narwhals-2.21.0-py3-none-any.whl (451 kB)



[notice] A new release of pip is available: 24.3.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
# ============================================
# IMPORT LIBRARIES
# ============================================

import numpy as np
import pandas as pd

import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import (
    StratifiedKFold,
    cross_val_score
)

from sklearn.compose import ColumnTransformer

from sklearn.pipeline import Pipeline

from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)

from sklearn.impute import SimpleImputer

from sklearn.metrics import accuracy_score

from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier
)

from xgboost import XGBClassifier

from lightgbm import LGBMClassifier

from catboost import CatBoostClassifier

In [4]:
# ============================================
# LOAD DATASET
# ============================================

train_df = pd.read_csv(
    "titanic/train.csv"
)

test_df = pd.read_csv(
    "titanic/test.csv"
)

In [5]:
# ============================================
# SAVE PASSENGER IDs
# ============================================

test_passenger_ids = test_df["PassengerId"]

In [6]:
# ============================================
# COMBINE DATASETS
# ============================================

full_df = pd.concat(
    [train_df, test_df],
    axis=0,
    ignore_index=True
)

In [7]:
# ============================================
# HANDLE MISSING VALUES
# ============================================

full_df["Age"] = full_df["Age"].fillna(
    full_df["Age"].median()
)

full_df["Fare"] = full_df["Fare"].fillna(
    full_df["Fare"].median()
)

full_df["Embarked"] = full_df["Embarked"].fillna(
    full_df["Embarked"].mode()[0]
)

full_df["Cabin"] = full_df["Cabin"].fillna(
    "Unknown"
)

In [8]:
# ============================================
# FAMILY SIZE
# ============================================

full_df["FamilySize"] = (
    full_df["SibSp"] +
    full_df["Parch"] + 1
)

In [9]:
# ============================================
# FAMILY TYPE
# ============================================

def family_type(size):
    
    if size == 1:
        return "Alone"
    
    elif size <= 4:
        return "Small"
    
    else:
        return "Large"

full_df["FamilyType"] = full_df[
    "FamilySize"
].apply(family_type)

In [10]:
# ============================================
# FARE PER PERSON
# ============================================

full_df["FarePerPerson"] = (
    full_df["Fare"] /
    full_df["FamilySize"]
)

In [11]:
# ============================================
# CHILD FEATURE
# ============================================

full_df["Child"] = (
    full_df["Age"] < 16
).astype(int)

In [12]:
# ============================================
# MOTHER FEATURE
# ============================================

full_df["Mother"] = (
    (full_df["Sex"] == "female") &
    (full_df["Parch"] > 0) &
    (full_df["Age"] > 18)
).astype(int)

In [13]:
# ============================================
# CABIN KNOWN FEATURE
# ============================================

full_df["CabinKnown"] = (
    full_df["Cabin"] != "Unknown"
).astype(int)

In [14]:
# ============================================
# DECK FEATURE
# ============================================

full_df["Deck"] = full_df[
    "Cabin"
].str[0]

full_df["Deck"] = full_df[
    "Deck"
].replace(
    ['G', 'T'],
    'Rare'
)

In [15]:
# ============================================
# TICKET FREQUENCY
# ============================================

full_df["TicketFrequency"] = full_df.groupby(
    "Ticket"
)["Ticket"].transform("count")

In [16]:
# ============================================
# TITLE FEATURE
# ============================================

full_df["Title"] = full_df["Name"].str.extract(
    ' ([A-Za-z]+)\.',
    expand=False
)

full_df["Title"] = full_df["Title"].replace(
    [
        'Lady','Countess','Capt','Col',
        'Don','Dr','Major','Rev',
        'Sir','Jonkheer','Dona'
    ],
    'Rare'
)

full_df["Title"] = full_df["Title"].replace(
    ['Mlle','Ms'],
    'Miss'
)

full_df["Title"] = full_df["Title"].replace(
    'Mme',
    'Mrs'
)

In [17]:
# ============================================
# DROP UNUSED FEATURES
# ============================================

full_df.drop(
    
    columns=[
        "PassengerId",
        "Name",
        "Ticket",
        "Cabin"
    ],
    
    inplace=True
)

In [18]:
# ============================================
# SPLIT DATASETS
# ============================================

train_processed = full_df[
    full_df["Survived"].notnull()
]

test_processed = full_df[
    full_df["Survived"].isnull()
]

In [19]:
# ============================================
# INPUT AND TARGET
# ============================================

X = train_processed.drop(
    "Survived",
    axis=1
)

y = train_processed["Survived"]

X_test = test_processed.drop(
    "Survived",
    axis=1
)

In [20]:
# ============================================
# FEATURE TYPES
# ============================================

categorical_features = X.select_dtypes(
    include=["object"]
).columns

numerical_features = X.select_dtypes(
    exclude=["object"]
).columns

In [21]:
# ============================================
# NUMERICAL PIPELINE
# ============================================

numeric_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        
        (
            "scaler",
            StandardScaler()
        )
    ]
)

In [22]:
# ============================================
# CATEGORICAL PIPELINE
# ============================================

categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

In [23]:
# ============================================
# COLUMN TRANSFORMER
# ============================================

preprocessor = ColumnTransformer(
    
    transformers=[
        
        (
            "num",
            numeric_transformer,
            numerical_features
        ),
        
        (
            "cat",
            categorical_transformer,
            categorical_features
        )
    ]
)

In [24]:
cat_model = CatBoostClassifier(
    
    iterations=2000,
    
    learning_rate=0.01,
    
    depth=5,
    
    loss_function='Logloss',
    
    verbose=0,
    
    random_state=42
)

In [25]:
xgb_model = XGBClassifier(
    
    n_estimators=1000,
    
    learning_rate=0.02,
    
    max_depth=3,
    
    subsample=0.8,
    
    colsample_bytree=0.8,
    
    random_state=42
)

In [26]:
lgbm_model = LGBMClassifier(
    
    n_estimators=1000,
    
    learning_rate=0.02,
    
    max_depth=3,
    
    random_state=42
)

In [27]:
# ============================================
# FULL MODEL PIPELINES
# ============================================

cat_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", cat_model)
    ]
)

xgb_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", xgb_model)
    ]
)

lgbm_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", lgbm_model)
    ]
)

In [28]:
# ============================================
# STRATIFIED K FOLD
# ============================================

skf = StratifiedKFold(
    
    n_splits=5,
    
    shuffle=True,
    
    random_state=42
)

In [29]:
cat_scores = cross_val_score(
    
    cat_pipeline,
    
    X,
    
    y,
    
    cv=skf,
    
    scoring="accuracy"
)

print("CatBoost Accuracy:")
print(cat_scores.mean())

CatBoost Accuracy:
0.8417111292448686


In [30]:
xgb_scores = cross_val_score(
    
    xgb_pipeline,
    
    X,
    
    y,
    
    cv=skf,
    
    scoring="accuracy"
)

print("XGBoost Accuracy:")
print(xgb_scores.mean())

XGBoost Accuracy:
0.8405938108091144


In [31]:
lgbm_scores = cross_val_score(
    
    lgbm_pipeline,
    
    X,
    
    y,
    
    cv=skf,
    
    scoring="accuracy"
)

print("LightGBM Accuracy:")
print(lgbm_scores.mean())

[LightGBM] [Info] Number of positive: 273, number of negative: 439
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001693 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 385
[LightGBM] [Info] Number of data points in the train set: 712, number of used features: 28
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.383427 -> initscore=-0.475028
[LightGBM] [Info] Start training from score -0.475028
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf

In [32]:
# ============================================
# TRAIN MODELS
# ============================================

cat_pipeline.fit(X, y)

xgb_pipeline.fit(X, y)

lgbm_pipeline.fit(X, y)

[LightGBM] [Info] Number of positive: 342, number of negative: 549
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000268 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 427
[LightGBM] [Info] Number of data points in the train set: 891, number of used features: 29
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.383838 -> initscore=-0.473288
[LightGBM] [Info] Start training from score -0.473288
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers co

In [33]:
# ============================================
# PREDICTION PROBABILITIES
# ============================================

cat_prob = cat_pipeline.predict_proba(
    X_test
)[:,1]

xgb_prob = xgb_pipeline.predict_proba(
    X_test
)[:,1]

lgbm_prob = lgbm_pipeline.predict_proba(
    X_test
)[:,1]

In [34]:
# ============================================
# BLEND PREDICTIONS
# ============================================

final_prob = (
    
    0.4 * cat_prob +
    
    0.3 * xgb_prob +
    
    0.3 * lgbm_prob
)

In [35]:
# ============================================
# FINAL PREDICTIONS
# ============================================

final_predictions = (
    final_prob >= 0.5
).astype(int)

In [36]:
# ============================================
# CREATE SUBMISSION FILE
# ============================================

submission = pd.DataFrame({
    
    "PassengerId": test_passenger_ids,
    
    "Survived": final_predictions
})

In [37]:
# ============================================
# SAVE SUBMISSION
# ============================================

submission.to_csv(
    "submission_advanced.csv",
    index=False
)

print("submission_advanced.csv created successfully!")

submission_advanced.csv created successfully!
